# Chat Models & Messages

**Prerequisites:** `01_your_first_llm.ipynb`

Notebook 1 made single, one-off calls. In reality every chat product feels continuous — the model "remembers" what you said five messages ago. It doesn't, actually. This notebook shows exactly what's really happening, then builds a small helper so you don't have to manage it by hand every time.

**By the end you'll have:**
- Used all three message roles (system / user / assistant) and seen the system role actually steer behavior
- Proven, not just been told, that LLM calls are stateless
- Built `Conversation`, a small helper in `chat.py` that manages history for you

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from llm import chat

## The three roles

Every message in the list has a `role`:

- **`system`** — instructions for how the model should behave. Set once, usually first.
- **`user`** — what the human says.
- **`assistant`** — what the model said back. You provide these too, when replaying history.

`system` isn't just a label — it actually changes behavior. Same question, two different system messages:

In [2]:
pirate = chat([
    {"role": "system", "content": "You are a pirate. Speak like one. Keep it to 2 sentences."},
    {"role": "user", "content": "Give me one tip for better sleep."},
])
formal = chat([
    {"role": "system", "content": "You are a formal doctor. Keep it to 2 sentences."},
    {"role": "user", "content": "Give me one tip for better sleep."},
])
print("PIRATE:", pirate)
print()
print("FORMAL:", formal)

PIRATE: Arr, dim the lanterns and stash yer gadgets so the night breeze can calm yer mind, matey. Then set a steady course to bed at the same hour each night, and ye’ll sail into dreamland with ease.

FORMAL: Adopt a consistent sleep–wake schedule by going to bed and arising at the same times each day, even on weekends; this regularity reinforces your circadian rhythm and promotes more restorative sleep.


Same user message, same model, same temperature — the only thing that changed is the system message, and the response is a completely different voice. That's what the system role is for.

## LLM calls are stateless — let's prove it

Nothing on the server remembers your last call. If you don't include something in `messages`, as far as the model is concerned it never happened. Let's test that directly: tell it a fact, then in a **separate call with no history**, ask it to recall that fact.

In [3]:
# First call: tell it something
chat([{"role": "user", "content": "My favorite color is teal."}])

# Second, unrelated call: ask it to recall
no_history = chat([{"role": "user", "content": "What color did I just say?"}])
print(no_history)

You didn’t mention a color in your last message, so there’s nothing for me to identify. If you have a specific color in mind, just let me know!


It has no idea — because that first call is gone the moment it returns. Now do the same thing, but include the earlier exchange as `user`/`assistant` messages in the same request:

In [4]:
with_history = chat([
    {"role": "user", "content": "My favorite color is teal."},
    {"role": "assistant", "content": "Got it, teal is a great choice!"},
    {"role": "user", "content": "What color did I just say?"},
])
print(with_history)

You mentioned **teal**.


Same model, same lack of server-side memory — the only difference is that the second request carried the whole conversation with it. "Conversation history" isn't a feature the API gives you; it's just a longer `messages` list you build and resend yourself, every single call. This is also why longer conversations cost more and get slower — you're re-sending everything, every time.

## Building history by hand

So a "conversation" is really just: append a user turn, call the model, append its reply, repeat.

In [5]:
messages = [{"role": "system", "content": "You are a concise assistant. Answer in one sentence."}]

messages.append({"role": "user", "content": "My favorite color is teal."})
reply = chat(messages)
messages.append({"role": "assistant", "content": reply})
print(reply)

Teal is a beautiful, calming shade!


In [6]:
messages.append({"role": "user", "content": "I also love hiking."})
reply = chat(messages)
messages.append({"role": "assistant", "content": reply})
print(reply)

That’s great—teal and the trail make a perfect combo for adventurous, nature‑loving vibes!


In [7]:
messages.append({"role": "user", "content": "Summarize what you know about me."})
reply = chat(messages)
messages.append({"role": "assistant", "content": reply})
print(reply)

You’re a teal‑enthusiast who enjoys hitting the trails and exploring nature through hiking.


It correctly references both teal *and* hiking — because both are sitting in `messages` by the time we ask. `len(messages)` is now 7: 1 system + 3 user + 3 assistant.

In [8]:
len(messages)

7

## From manual history to a reusable wrapper

The append/call/append pattern above is always identical, so `chat.py` wraps it in a small `Conversation` class:

```python
from llm import chat


class Conversation:
    def __init__(self, system=None):
        self.messages = []
        if system:
            self.messages.append({"role": "system", "content": system})

    def send(self, content, **kwargs):
        self.messages.append({"role": "user", "content": content})
        reply = chat(self.messages, **kwargs)
        self.messages.append({"role": "assistant", "content": reply})
        return reply
```

In [9]:
from chat import Conversation

convo = Conversation(system="You are a concise assistant. Answer in one sentence.")
print(convo.send("My favorite color is teal."))
print(convo.send("I also love hiking."))
print(convo.send("Summarize what you know about me."))

Teal is a beautiful, calming shade that blends the serenity of blue with the energy of green.


Hiking is a wonderful way to explore nature and discover new vistas.


You enjoy the calming hue of teal and love exploring nature through hiking.


Same result as building the list by hand, minus the bookkeeping. `convo.messages` still holds the full history if you need to inspect it.

## What you built

✓ Seen the system role actually change model behavior, side by side

✓ Proved statelessness directly — a fact recalled only when it's in `messages`, forgotten otherwise

✓ Built conversation history by hand, then wrapped it as `Conversation` in `chat.py`

**Next:** `03_tokens_and_context.ipynb` — why that growing `messages` list isn't free, and what happens when it gets too big.